# Week 1, Day 5: A Hall of Fame tribute in two calls

**Lesson:** [week1/day5.ipynb](../../../week1/day5.ipynb), *A full business solution: the brochure generator*. **About 45 minutes.**

**What you practice here**

- JSON mode: a one-shot prompt that shows the JSON you want, a `response_format` that asks for a JSON object, and `json.loads` on the reply.
- Chaining calls: the first call's JSON decides which pages to fetch, and a second call writes from what was fetched.
- Streaming: asking for the reply in chunks, and building the text from each chunk's new piece as it arrives.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week1/day5", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson builds a company brochure in two model calls. The first reads the links on a website and answers in JSON
with the ones worth fetching. The second writes the brochure from those pages, streamed so it appears as it is
written. Do the same for a wrestling Hall of Fame. From a wrestler's Wikipedia page, the first call picks the pages
that matter for a tribute (championships, famous matches, rivals). The second streams a short induction tribute
written from them.

The functions take the client as their first parameter. `tribute` also takes the page fetcher, which is the lesson's
`fetch_website_contents` when you run it for real. The checks hand them stand-ins that record what you send and
answer with canned replies.

## What each call should give

```python
select_links(client, "https://en.m.wikipedia.org/wiki/Bret_Hart", links)
# one request with the model gpt-5-nano, asking for a JSON object; returns the parsed JSON, such as
# {'links': [{'type': 'championship page', 'url': 'https://en.m.wikipedia.org/wiki/WWE_Championship'},
#            {'type': 'famous match page', 'url': 'https://en.m.wikipedia.org/wiki/Montreal_Screwjob'}]}

for text in stream_tribute(client, "Bret Hart", pages):
    print(text)
# one streamed request with the model gpt-4.1-mini; it yields the text so far, longer each time:
# '# Bret "The'
# '# Bret "The Hitman" Hart'
# '# Bret "The Hitman" Hart\n\nThe best'   ... and so on, until the whole tribute

tribute(client, "Bret Hart", url, links, fetch_website_contents)
# picks the links, fetches the main page and each pick, then yields what stream_tribute yields
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.
import json

### 1 · `select_links(client, url, links, model="gpt-5-nano")`

Asks the model in JSON mode which links matter for a tribute, and returns its JSON as a dict. `links` are the links
found on the page at `url`, as the lesson's `fetch_website_links` returns them, so many are relative, like
`/wiki/WWE_Championship`. Make one Chat Completions call with the model `model`, a response format that asks for a
JSON object, and two messages:

- a system message of your own. It asks for the links most relevant to a tribute (championships, famous matches,
  rivals), at most three of them, and a reply in JSON. It shows a one-shot example in the shape
  `{"links": [{"type": "championship page", "url": "https://..."}]}`, so `"links"`, `"type"` and `"url"` appear in it
  with their double quotes.
- a user message that names `url`, asks for full `https` URLs, then lists every link on a line of its own.

Return the reply's text parsed from JSON into a dict.

<details><summary>Hint</summary>

```python
response = client.chat.completions.create(model=model, messages=[...],
                                          response_format={"type": "json_object"})
return json.loads(response.choices[0].message.content)
```

</details>

In [ ]:
def select_links(client, url, links, model="gpt-5-nano"):
    # Write select_links here, as described above.
    raise NotImplementedError("select_links is not written yet")

In [ ]:
check("select_links")

### 2 · `stream_tribute(client, wrestler, pages, model="gpt-4.1-mini")`

Streams a Hall of Fame tribute written from the pages' text, yielding the text so far as it grows. Write it as a
generator: a function that uses `yield`. Generators are new here (the course uses them more from Week 2), and Part 2
of `guides/10_intermediate_python.ipynb` explains them if you need it. Make one Chat Completions call with the model
`model` and streaming on. Send a system message of your own (a short tribute, in markdown without code blocks), then
a user message that starts by naming `wrestler` and then holds `pages`. Cut the whole user message to its first
5,000 characters, as the lesson's `get_brochure_user_prompt` does: as the name comes first, it survives the cut. For
each chunk, add the chunk's new text to the text so far, and yield the text so far. A chunk's text can be `None`,
which the lesson's `or ''` turns into an empty string. This is the lesson's loop, with `yield` where it had
`update_display`.

<details><summary>Hint</summary>

```python
stream = client.chat.completions.create(model=model, messages=[...], stream=True)
for chunk in stream:
    text += chunk.choices[0].delta.content or ""
    yield text
```

</details>

In [ ]:
def stream_tribute(client, wrestler, pages, model="gpt-4.1-mini"):
    # Write stream_tribute here, as described above.
    raise NotImplementedError("stream_tribute is not written yet")

In [ ]:
check("stream_tribute")

### 3 · `tribute(client, wrestler, url, links, fetch)`

Chains the two calls: picks the links, fetches the main page and each picked page, then streams the tribute.
`fetch(url)` returns a page's text; when you run it for real, it is the lesson's `fetch_website_contents`. Call
`select_links(client, url, links)`. Then fetch the main page, `url`, and then the `"url"` of each picked link, in
order, and fetch nothing else. Join the texts into one, with each picked page under a heading that gives its
`"type"`, as the lesson's `fetch_page_and_all_relevant_links` does. Then yield what
`stream_tribute(client, wrestler, pages)` yields.

<details><summary>Hint</summary>

```python
for link in select_links(client, url, links)["links"]:
    ...  # link["type"] and link["url"]
yield from stream_tribute(client, wrestler, pages)
```

</details>

In [ ]:
def tribute(client, wrestler, url, links, fetch):
    # Write tribute here, as described above.
    raise NotImplementedError("tribute is not written yet")

In [ ]:
check("tribute")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 1, Day 5 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A tribute takes two calls, a cent or two at most, and fetches each picked page from Wikipedia.

1. Load your key with `load_dotenv(override=True)` and make `client = OpenAI()`. The lesson's scraper is in
   `week1/scraper.py`, so put that folder on `sys.path` (`sys.path.insert(0, str(ROOT / "week1"))`), then import
   `fetch_website_links` and `fetch_website_contents` from `scraper`.
2. Take a wrestler's mobile Wikipedia page, such as `url = "https://en.m.wikipedia.org/wiki/Bret_Hart"` (desktop pages
   start with menus). Print the first 500 characters of `fetch_website_contents(url)`: the start of what the model
   will read about the wrestler.
3. Make `links = fetch_website_links(url)` and print `select_links(client, url, links)`. It should pick at most three:
   the tribute's prompt is cut to 5,000 characters, which holds the main page and one or two picks.
4. Stream the tribute into the notebook as the lesson does: `handle = display(Markdown(""), display_id=True)`, then
   `update_display(Markdown(text), display_id=handle.display_id)` for each text from
   `tribute(client, "Bret Hart", url, links, fetch_website_contents)`.
5. Make the tribute's system prompt humorous, run its cell again, and stream a second tribute, as the lesson does
   with its brochure.

<details><summary>Stuck? One way to write it</summary>

```python
import sys

from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI()
sys.path.insert(0, str(ROOT / "week1"))  # the lesson's scraper.py
from scraper import fetch_website_contents, fetch_website_links

url = "https://en.m.wikipedia.org/wiki/Bret_Hart"
print(fetch_website_contents(url)[:500])  # the start of what the model reads

links = fetch_website_links(url)
print(select_links(client, url, links))

handle = display(Markdown(""), display_id=True)
for text in tribute(client, "Bret Hart", url, links, fetch_website_contents):
    update_display(Markdown(text), display_id=handle.display_id)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.